In [2]:
!pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 MB 9.0 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 19.1 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.5-py2.py3-none-any.whl size=317747923 sha256=3f6641db1e948a0c32308aecf47f6b1e76a9dd3d16f21d97c8d67841439ae8b7
  Stored in directory: /Users/junpark/Library/Caches/pip/wheels/8f/cb/c0/cc57eb1bf0f9dc87cdaf2b0dbac49e58a210ff68d21d6fc709
Successfully built pyspark

[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import pyspark

In [5]:
pyspark.__version__

'3.5.5'

In [6]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("test").getOrCreate()

df = spark.read.option("header", "true").parquet("yellow_tripdata_2024-10.parquet")

df.show()

25/03/03 11:52:14 WARN Utils: Your hostname, Jun-M1Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.45.66 instead (on interface en0)
25/03/03 11:52:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/03 11:52:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-10-01 00:30:44|  2024-10-01 00:48:26|              1|          3.0|         1|                 N|         162|         246|           1|       18.4|  1.0|    0.5|       1.

In [7]:
df.write.parquet("test")

In [8]:
""" 
Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.
"""

df.repartition(4).write.parquet("q2")

In [9]:
!ls -lh q2

total 188928
-rw-r--r--@ 1 junpark  staff     0B Mar  3 11:57 _SUCCESS
-rw-r--r--@ 1 junpark  staff    22M Mar  3 11:57 part-00000-08d8d2f5-a292-49dd-9b4c-6287ec1f3634-c000.snappy.parquet
-rw-r--r--@ 1 junpark  staff    22M Mar  3 11:57 part-00001-08d8d2f5-a292-49dd-9b4c-6287ec1f3634-c000.snappy.parquet
-rw-r--r--@ 1 junpark  staff    22M Mar  3 11:57 part-00002-08d8d2f5-a292-49dd-9b4c-6287ec1f3634-c000.snappy.parquet
-rw-r--r--@ 1 junpark  staff    22M Mar  3 11:57 part-00003-08d8d2f5-a292-49dd-9b4c-6287ec1f3634-c000.snappy.parquet


In [29]:
"""
How many taxi trips were there on the 15th of October?

Consider only trips that started on the 15th of October.
"""

from pyspark.sql import functions as F

df.filter(
    (F.dayofmonth(df["tpep_pickup_datetime"]) == 15)
    & (F.month(df["tpep_pickup_datetime"]) == 10)
).count()

128893

In [25]:
df.select(F.dayofmonth(df["tpep_pickup_datetime"])).show()

+--------------------------------+
|dayofmonth(tpep_pickup_datetime)|
+--------------------------------+
|                               1|
|                               1|
|                               1|
|                               1|
|                               1|
|                               1|
|                               1|
|                               1|
|                               1|
|                               1|
|                               1|
|                               1|
|                               1|
|                               1|
|                              30|
|                               1|
|                               1|
|                               1|
|                               1|
|                               1|
+--------------------------------+
only showing top 20 rows



In [40]:
### What is the length of the longest trip in the dataset in hours?

df.agg(
    F.max(
        (
            F.unix_timestamp(df["tpep_dropoff_datetime"])
            - F.unix_timestamp(df["tpep_pickup_datetime"])
        )
        / 3600
    )
).show()

+--------------------------------------------------------------------------------------------------------------------------------------+
|max(((unix_timestamp(tpep_dropoff_datetime, yyyy-MM-dd HH:mm:ss) - unix_timestamp(tpep_pickup_datetime, yyyy-MM-dd HH:mm:ss)) / 3600))|
+--------------------------------------------------------------------------------------------------------------------------------------+
|                                                                                                                    162.61777777777777|
+--------------------------------------------------------------------------------------------------------------------------------------+



In [48]:
### Using the zone lookup data and the Yellow October 2024 data, what is the name of the LEAST frequent pickup location Zone?

zone_df = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)

df.join(zone_df, df["PULocationID"] == zone_df["LocationID"], "left").groupby(
    "Zone"
).count().orderBy("count").show()

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Governor's Island...|    1|
|       Rikers Island|    2|
|       Arden Heights|    2|
|         Jamaica Bay|    3|
| Green-Wood Cemetery|    3|
|Charleston/Totten...|    4|
|   Rossville/Woodrow|    4|
|       West Brighton|    4|
|Eltingville/Annad...|    4|
|       Port Richmond|    4|
|         Great Kills|    6|
|        Crotona Park|    6|
|Heartland Village...|    7|
|     Mariners Harbor|    7|
|Saint George/New ...|    9|
|             Oakwood|    9|
|       Broad Channel|   10|
|New Dorp/Midland ...|   10|
|         Westerleigh|   12|
|     Pelham Bay Park|   12|
+--------------------+-----+
only showing top 20 rows

